# Door 3: a trained classifier, and what it honestly buys you

This notebook accompanies the docs page [`door3-classifier`](../../docs/examples/door3-classifier.md). A two-component Gaussian mixture stands in for a likelihood ScoreQuant cannot evaluate: only a classifier trained to separate labeled signal and background events is available. The notebook trains that classifier, converts its calibrated posteriors into density ratios and scores with `DensityRatioScore.from_classifier`, fits a quantizer, and runs a longer classifier-quality ladder than the docs page's fast snippets use, comparing self-reported retention against the truth.

## Data

`examples.door3_classifier` holds the mixture definition, the classifier trainer, and the exact-score oracle shared with the docs page and its committed figure. Ladder and sample sizes shrink under `SCOREQUANT_EXAMPLE_FAST` through `example_scale`.

In [ ]:
import scorequant as sq
from examples._env import example_scale
from examples.door3_classifier import (
    REFERENCE_FRACTIONS,
    classifier_provider,
    draw_reference_mixture,
    exact_provider,
    make_figure,
    run_ladder,
    train_classifier,
)

n_train = example_scale(3_000, 400)
n_test = example_scale(4_000, 600)
train_observations = draw_reference_mixture(2026, n_train)
test_observations = draw_reference_mixture(999, n_test)
REFERENCE_FRACTIONS, train_observations.shape, test_observations.shape

## Train, wrap, and fit

Training and calibration stay application code; ScoreQuant starts at `predict_proba`, the declared training priors, and a declared parameterization. `information_kind` reads `"supplied_score_surrogate"` for every classifier-derived result, regardless of how good the classifier is.

In [ ]:
model = train_classifier(seed=101, n_per_class=example_scale(150, 40))
provider = classifier_provider(model, description="logistic regression")
result = sq.fit_quantizer(
    sq.ObservationSample(train_observations),
    provider=provider,
    n_bins=4,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=7, initializer_restarts=8),
)
result.information_kind

## The retention ladder

`run_ladder` refits at three labeled training-set sizes and reports both the surrogate retention (`evaluate_scores` on the estimated scores themselves) and the true retention (the exact oracle score, evaluated at the labels the estimated-score quantizer actually produced).

In [ ]:
ladder_sizes = (example_scale(30, 10), example_scale(150, 40), example_scale(1_500, 150))
steps = run_ladder(
    n_per_class_values=ladder_sizes,
    n_train=n_train,
    n_test=n_test,
    n_bins=4,
)
[
    (step.n_per_class, round(step.surrogate_retention, 4), round(step.true_retention, 4))
    for step in steps
]

In [ ]:
fig = make_figure(steps)
fig

## The oracle ceiling

Fitting directly from the exact score — never estimated — shows what the ladder is converging toward: the same D-optimal ceiling, reached once the classifier is good enough that its surrogate number can finally be trusted.

In [ ]:
oracle = exact_provider()
oracle_result = sq.fit_quantizer(
    sq.ObservationSample(train_observations),
    provider=oracle,
    n_bins=4,
    criterion=sq.DOptimality(),
    config=sq.DExchangeConfig(seed=7, initializer_restarts=8),
)
oracle_test_report = oracle_result.evaluate_scores(oracle.score(test_observations))
oracle_result.information_kind, float(oracle_test_report.geometric_mean_retention)

## Interpretation

The surrogate retention barely moves across the ladder — the D-optimal solver is genuinely good at preserving whatever score it is handed, estimated or not. The true retention starts far below it at the smallest training size and climbs toward the oracle ceiling as the classifier improves. The gap between the two is exactly what `information_kind` is warning about: a classifier-backed result's own diagnostics describe the estimated score, not the one a downstream fit actually depends on.